# Pipeline Walkthrough - trading-BEI

An end-to-end tour of the pipeline, one step at a time, with the tensor shapes printed at
every stage and a **real training loop** at the end.

```
parquet -> features(15) -> pick groups -> normalize -> window into tensors
        -> model -> TRAIN -> honest backtest (T+1 execution, spread costs) vs IHSG
```

Run top to bottom. Steps 1-8 explore the data/tensors; steps 9-13 actually train a model and
backtest it. Uses a recent, capped slice so everything runs in seconds/minutes (the real sweep
in `compare.py` uses the full universe).

**Execution-timing convention (important).** Features at day *t* use day-*t* CLOSING data, so an
order informed by them cannot fill at that same close. Everything downstream is built around
`execution_lag = 1`: decide after the close of *t*, **enter at the close of t+1, exit at the close
of t+2**. Labels, training, and the simulator all share this convention — a model evaluated any
other way is measuring a trade you cannot place.

In [ ]:
import os, sys
# run from the repo root so `src` imports and data/ paths resolve
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
import torch
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('cwd   :', os.getcwd())
print('device:', device, '(training cells will use this)')

## 1. Load the panel (raw parquet)

The single source of truth. **Long format**: one row per (date, stock). Everything downstream is
derived from this file.

In [ ]:
panel = pd.read_parquet('data/processed/panel.parquet')
print('shape :', panel.shape, '  (rows, columns)')
print('dates :', panel['date'].min().date(), '->', panel['date'].max().date())
print('stocks:', panel['ticker'].nunique())
print('columns:', list(panel.columns))
panel.head(3)

## 2. Peek at one stock's raw rows

Data-quality reality of IDX: on non-trading days rows still exist but carry zeros
(open/high/low), and `OpenPrice` is 0 most of the time. The `bid_volume` / `offer_volume` /
`weight_for_index` columns are the ones we added to the scraper - they feed feature group **G5**
(microstructure) and the **IHSG** benchmark.

In [ ]:
cols = ['date','ticker','prev_close','open','high','low','close','volume','value',
        'bid','bid_volume','offer','offer_volume','weight_for_index']
tk = 'BBCA' if (panel['ticker'] == 'BBCA').any() else panel['ticker'].iloc[0]
print('example ticker:', tk)
panel[panel['ticker'] == tk][cols].tail(5)

## 3. compute_features - the full 15-feature superset

Turns raw prices/volumes into **15 causal features** (no look-ahead) plus the label
`fwd_return` = the return a signal at day *t* can actually capture: **enter at close t+1, exit at
close t+2** (`execution_lag=1`). The label only exists when t+1/t+2 are strictly consecutive
exchange days and the stock really trades on both — a "return" across a suspension gap or off a
stale print never becomes a training label.

Key design choice: rows are dropped **only** where the core feature `log_return` is NaN. Any other
feature that is NaN for a row (sparse bid/offer, rolling warm-up) is kept and later neutralized to
0 - so the set of samples is identical no matter which feature groups an experiment turns on.

In [ ]:
from src.preprocess import compute_features, ALL_FEATURES, TARGET_COLUMN, FEATURE_GROUPS

# execution_lag=1 is the default; spelled out here because it defines what the label MEANS:
# fwd_return at t = log return from close(t+1) to close(t+2)
feats = compute_features(panel, horizon=1, execution_lag=1)
print('feats shape:', feats.shape)
print('15 features:', ALL_FEATURES)
print('target     :', TARGET_COLUMN)
print()
# NaN rate per feature (these get neutralized to 0 in normalize, not dropped)
feats[ALL_FEATURES].isna().mean().round(3).to_frame('nan_rate')

In [ ]:
# the feature values for one stock (raw, before normalization)
feats[feats['ticker'] == tk][['date', *ALL_FEATURES, TARGET_COLUMN]].tail(4)

## 4. Feature groups & config resolution

The ablation study turns features on/off by **economic group**, not one-by-one. `resolve_features`
maps a config's `feature_groups` list to the concrete active columns.

| Group | Theme |
|-------|-------|
| G1 | return & momentum |
| G2 | volatility / range |
| G3 | volume & liquidity |
| G4 | foreign flow |
| G5 | microstructure (bid/offer) |

In [ ]:
from src.preprocess import resolve_features

for g, colnames in FEATURE_GROUPS.items():
    print(f'{g}: {colnames}')
print()
print('Exp A (G1)        ->', resolve_features({'feature_groups': ['G1']}))
print('Exp C (G1,G2,G3)  ->', resolve_features({'feature_groups': ['G1','G2','G3']}))
print('full (all)        ->', resolve_features(None))

## 5. normalize - cross-sectional z-score per day

For each **day**, standardize each feature across the stocks trading that day. It uses only that
day's cross-section, so it is causal (no train-fit stats to leak). Missing values become 0
(neutral). Sanity check below: on any day the mean should be ~0 and std ~1
(std < 1 for heavy-tailed features like `turnover`/`amihud` because of the outlier clip).

In [ ]:
from src.preprocess import normalize

active = resolve_features({'feature_groups': ['G1','G2','G3']})   # 11 features (Exp C)
feats_n = normalize(feats, columns=active)

d = feats_n['date'].iloc[len(feats_n) // 2]
day = feats_n[feats_n['date'] == d][active]
print('day:', pd.Timestamp(d).date(), '| stocks that day:', len(day))
print()
print('mean (should be ~0):'); print(day.mean().round(3).to_dict())
print()
print('std  (should be ~1):'); print(day.std(ddof=0).round(3).to_dict())

## 6. Windowing into tensors - per-stock dataset

`IDXWindowDataset` slides a `lookback`-day window per stock. One **sample** = one (stock, day):
an `x` matrix of shape `(lookback, n_features)` and a scalar label `y` = the return that stock
delivers from close(t+1) to close(t+2) — what a signal formed at t can capture. This is what a
plain per-stock model consumes.

In [ ]:
from src.dataset import IDXWindowDataset

# recent slice so datasets build fast (keeps history for the 60-day lookback)
feats_recent = feats_n[feats_n['date'] >= pd.Timestamp('2024-06-01')]
lookback = 60
ds = IDXWindowDataset(feats_recent, lookback=lookback, feature_cols=active)
print('samples:', len(ds), '| n_features:', ds.n_features)

x, y, meta = ds[0]
print()
print('one sample:')
print('  x shape:', tuple(x.shape), '= (lookback, n_features)')
print('  y (log return close t+1 -> t+2):', round(float(y), 5))
print('  meta:', meta)
print()
print('x[:3]  (first 3 timesteps, already normalized):')
print(x[:3])

## 7. Cross-sectional dataset - one day = all stocks

`IDXCrossSectionalDataset` returns a whole day at once, so the flagship model can attend **across
stocks** (compare a stock to the rest of the market that day). One sample = one day: `X` of shape
`(N_stocks, lookback, n_features)`. N varies by day, so days are processed one at a time.

In [ ]:
from src.dataset_cs import IDXCrossSectionalDataset

ds_cs = IDXCrossSectionalDataset(feats_recent, lookback=lookback, feature_cols=active, min_stocks=20)
print('days:', len(ds_cs))

X, yv, tickers, date = ds_cs[len(ds_cs) // 2]
print()
print('one day:')
print('  date    :', pd.Timestamp(date).date())
print('  X shape :', tuple(X.shape), '= (N_stocks, lookback, n_features)')
print('  y shape :', tuple(yv.shape), '= (N_stocks,)')
print('  N stocks:', len(tickers), '| first 5:', tickers[:5])

## 8. Model forward pass (untrained)

Tensors in -> one score per stock out. Ranking the scores each day drives the top-N buys.
Weights are **random** here, so the scores mean nothing yet - this only shows input/output shapes.

In [ ]:
from models.transformer import TransformerPolicy
from models.cross_sectional import CrossSectionalModel

torch.manual_seed(0)
per_stock = TransformerPolicy(n_features=len(active), d_model=64, n_heads=4, n_layers=3,
                              dim_ff=128, dropout=0.1, lookback=lookback, output='linear').eval()
batch = torch.stack([ds[i][0] for i in range(8)])          # (8, lookback, F)
with torch.no_grad():
    print('per-stock  : in', tuple(batch.shape), '-> out', tuple(per_stock(batch).shape))

torch.manual_seed(0)
cross = CrossSectionalModel(n_features=len(active), d_model=64, n_heads=4,
                            temporal_layers=2, cross_layers=2, dim_ff=128,
                            dropout=0.1, lookback=lookback, output='linear').eval()
with torch.no_grad():
    print('cross-sect : in', tuple(X.shape), '-> out', tuple(cross(X).shape), '= one score per stock')
print('per-stock params:', sum(p.numel() for p in per_stock.parameters()))

## 8b. The round trip: tensor in -> scores out -> back into one panel

The NN never sees "the whole dataset" at once. It sees **one day at a time**:

```
IN   X  (N_stocks, lookback, n_features)   e.g. (241, 60, 11)
           |  each of the N stocks: its own last-60-day history of 11 features
           v
        model
           |
OUT  s  (N_stocks,)                        one scalar SCORE per stock, that day
```

The score has no unit — it only needs to *rank* stocks within the day (top-10 get bought).

N changes every day (listings, suspensions, the liquidity screen), so the outputs can't be
stacked into one rectangular tensor. Instead we go back to the same trick the input data uses:
**long format**. Loop over days, and for each day append rows `(date, ticker, score)`. That long
DataFrame — the "score panel" — is exactly what `predict_scores_cs` returns and what the
backtest consumes. Pivot it if you want the `date x ticker` matrix view (holes = stock not
scoreable that day).

In [ ]:
# the exact journey, on 3 example days (model still untrained -- shapes are the point)
rows = []
with torch.no_grad():
    for i in range(3):
        X, yv, tickers, date = ds_cs[i]
        s = cross(X)                                     # (N,) one score per stock
        print(f'{pd.Timestamp(date).date()}   in X {tuple(X.shape)}  ->  out scores {tuple(s.shape)}')
        for tk, sc in zip(tickers, s.numpy()):
            rows.append((pd.Timestamp(date), tk, float(sc)))

scores_long = pd.DataFrame(rows, columns=['date', 'ticker', 'score'])
print()
print('collected LONG format (= what predict_scores_cs returns, what the backtest eats):')
print(scores_long.head(6).to_string(index=False))
print('...', len(scores_long), 'rows total = sum of N over the 3 days')

# same information pivoted into a date x ticker score matrix
score_matrix = scores_long.pivot(index='date', columns='ticker', values='score')
print()
print('pivoted matrix view:', score_matrix.shape, '= (days, union of tickers); NaN = not scoreable that day')
score_matrix.iloc[:3, :6]

## 9. Training - one explicit gradient step

Before a full loop, here is the mechanic in isolation on a single batch:

1. forward -> prediction, 2. MSE loss vs the true next-day return, 3. `backward()` fills gradients,
4. `opt.step()` nudges the weights down the gradient.

The loss on the **same batch** should drop after one step - proof that gradients flow and the
optimizer works.

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
net = TransformerPolicy(n_features=len(active), lookback=lookback, output='linear').to(device)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

xb = torch.stack([ds[i][0] for i in range(64)]).to(device)   # (64, lookback, F)
yb = torch.stack([ds[i][1] for i in range(64)]).to(device)   # (64,)

before = loss_fn(net(xb), yb).item()
opt.zero_grad()
loss = loss_fn(net(xb), yb)
loss.backward()                                              # 3) compute gradients
gnorm = sum(p.grad.norm().item() for p in net.parameters() if p.grad is not None)
opt.step()                                                   # 4) update weights
after = loss_fn(net(xb), yb).item()

print(f'loss before step : {before:.6f}')
print(f'grad norm        : {gnorm:.4f}   (nonzero -> gradients flowed)')
print(f'loss after step  : {after:.6f}   (lower on the same batch -> it learned)')

## 10. Training loop (per-stock, MSE) - watch the loss go down

Now the full loop over several epochs: shuffle -> minibatch -> forward/backward/step -> then
evaluate on a held-out (later) window. To keep the notebook fast we cap to the **200 most-liquid
stocks**. NOTE: this cap ranks by average value over the *whole* panel — a demo shortcut with
mild look-ahead that is fine for a mechanics walkthrough but would be cheating in research; the
real pipeline uses the causal trailing-20d screen in `market.apply_universe`.

In [ ]:
from torch.utils.data import DataLoader

# cap universe to the 200 most-liquid names (by avg traded value) -> fast, sensible demo
liquid = panel.groupby('ticker')['value'].mean().sort_values(ascending=False).head(200).index
feats_liquid = feats_recent[feats_recent['ticker'].isin(liquid)]
print('capped universe:', feats_liquid['ticker'].nunique(), 'stocks')

train_ds = IDXWindowDataset(feats_liquid, lookback, end='2025-03-31', feature_cols=active)
val_ds   = IDXWindowDataset(feats_liquid, lookback, start='2025-04-01', end='2025-05-31', feature_cols=active)
print('train samples:', len(train_ds), '| val samples:', len(val_ds))

def collate(b):
    return torch.stack([r[0] for r in b]), torch.stack([r[1] for r in b]), [r[2] for r in b]

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, collate_fn=collate, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=256, shuffle=False, collate_fn=collate, num_workers=0)

torch.manual_seed(0)
net = TransformerPolicy(n_features=len(active), d_model=64, n_heads=4, n_layers=2,
                        dim_ff=128, dropout=0.1, lookback=lookback, output='linear').to(device)
opt = torch.optim.Adam(net.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

for epoch in range(3):
    net.train(); tot = n = 0
    for xb, yb, _ in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(net(xb), yb)
        loss.backward()
        opt.step()
        tot += loss.item() * len(yb); n += len(yb)
    net.eval(); vt = vn = 0
    with torch.no_grad():
        for xb, yb, _ in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            vt += loss_fn(net(xb), yb).item() * len(yb); vn += len(yb)
    print(f'epoch {epoch} | train MSE {tot/n:.6f} | val MSE {vt/vn:.6f}')

## 11. The real flagship training (cross-sectional Sharpe objective)

This is exactly what `compare.py` runs, just on a tiny window here. `train_dlsa` does **not** minimize
MSE - each day it turns scores into long-only portfolio weights (softmax), computes the daily
portfolio return, and maximizes the **Sharpe** of those returns over a mini-batch of days. Watch
`val_sharpe` (higher is better) and the per-epoch seconds (from the timing we added).

In [ ]:
from src import train_test as tt

# small walk-forward split on the capped universe (real sweep: 2022-01 .. 2026-07)
tr = IDXCrossSectionalDataset(feats_liquid, lookback, start='2025-01-01', end='2025-04-30', feature_cols=active, min_stocks=20)
va = IDXCrossSectionalDataset(feats_liquid, lookback, start='2025-05-01', end='2025-05-31', feature_cols=active, min_stocks=20)
te = IDXCrossSectionalDataset(feats_liquid, lookback, start='2025-06-01', feature_cols=active, min_stocks=20)
print('days -> train:', len(tr), 'val:', len(va), 'test:', len(te))

torch.manual_seed(0)
flagship = CrossSectionalModel(n_features=len(active), d_model=64, n_heads=4,
                               temporal_layers=2, cross_layers=2, dim_ff=128,
                               dropout=0.1, lookback=lookback, output='linear').to(device)
cfg = {'lr': 3e-4, 'weight_decay': 1e-5, 'epochs': 4, 'early_stop_patience': 4,
       'days_per_step': 16, 'softmax_temp': 1.0}
print()
flagship = tt.train_dlsa(flagship, tr, va, cfg, device=device)

## 12. IHSG proxy benchmark

Cap-weighted market return (buy & hold) reconstructed from the panel - the minimum bar a long-only
strategy must beat (a high Sharpe can still lose to just holding the market).

Weighting matters: the cap is `close × weight_for_index` (IDX's float-adjusted share count).
Using `weight_for_index` *directly* as the weight was a real bug once shipped here — it
share-weights the index, letting trillion-share penny stocks (GOTO alone ~21%) dominate, and
showed +53% over a window where the real IHSG fell ~20%. The fixed proxy tracks the real index
to within ~1pp per year.

In [ ]:
from src.benchmark import ihsg_proxy_returns, benchmark_metrics

ihsg = ihsg_proxy_returns(panel)
print('IHSG proxy daily returns (tail):')
print(ihsg.tail(3))

## 13. Backtest the TRAINED model - honest simulator vs IHSG

Two tiers, exactly like `run_train_test`:

- **Quick label-based backtest** (`tt.backtest_long_only`) - frictionless, used only for
  *validation-time model selection* where speed matters and only the ranking of models counts.
- **The real evaluation** (`simulate_long_only`) - scores dated *t* fill at the close of **t+1**
  (`execution_lag=1`), names pinned at ARA can't be bought / at ARB can't be sold, suspended
  holdings stay in the book, and every fill pays commission **plus that name's own half-spread**
  from its closing book. This is the number that means something.

(Tiny, capped demo - the real conclusion comes from the full `compare.py` sweep.)

In [ ]:
from src.market import build_market
from src.backtest import simulate_long_only, benchmark_relative_metrics

scores = tt.predict_scores_cs(flagship, te, device=device)

# (a) quick label-based backtest -- selection metric only, not the result
quick, _ = tt.backtest_long_only(scores, top_n=10)
print(f"quick label-based net sharpe: {quick['sharpe']:.2f}   (frictionless; for model selection)")

# (b) the REAL evaluation: T+1 fills, ARA/ARB tradability, commission + half-spread
market = build_market(panel)          # adjusted returns + tradability + half_spread
metrics, daily = simulate_long_only(scores[['date', 'ticker', 'score']], market, top_n=10)
metrics.update(benchmark_relative_metrics(daily, ihsg))

bm = benchmark_metrics(ihsg, dates=daily['date'])
print()
print('=== TRAINED flagship, honest long-only top-10 (test) ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print()
print(f"strategy ann_return: {metrics['ann_return']:.2%}")
print(f"IHSG     ann_return: {bm['ihsg_ann_return']:.2%}")
print('  ->', 'BEATS' if metrics['ann_return'] > bm['ihsg_ann_return'] else 'LOSES TO',
      'IHSG (tiny demo - not conclusive)')
daily.head()

---

### Flow summary

| Step | Call | Output |
|------|------|--------|
| 1 | `read_parquet` | raw (date, stock) rows |
| 3 | `compute_features` | 15 causal features + `fwd_return` (close t+1 -> t+2, `execution_lag=1`) |
| 4 | `resolve_features` | active columns from group config |
| 5 | `normalize` | cross-sectional z-score per day |
| 6-7 | `IDXWindowDataset` / `IDXCrossSectionalDataset` | tensors `(lookback, n_features)` |
| 8 | model | one score per stock |
| 9-11 | gradient step / MSE loop / `train_dlsa` | a trained model |
| 12 | `ihsg_proxy_returns` | buy & hold benchmark (cap = close x weight_for_index) |
| 13 | `build_market` + `simulate_long_only` | honest backtest: T+1 fills, ARA/ARB, spread costs |

For the real experiment (all groups A-E x 8 seeds, full universe, walk-forward):

```
python compare.py --seeds 8        # sequential: one run needs ~5-8 GB VRAM (8 GB card -> no --parallel)
# or a single config:
python -m src.run_train_test -c configs/cross_sectional.yaml
```